# JobInterviewGuide Workshop — Targeted Practice Notebook

Style inspired by the workshop notebooks (talking points + clear steps + sanity checks).

## What this notebook is for
This notebook focuses on topics that were missed in the quiz:
- Data leakage + correct Train/Val/Test usage
- Metrics for imbalanced classification (Precision/Recall, PR-AUC, F1)
- Linear regression evaluation (MSE, $R^2$) and overfitting vs labeling errors
- Logistic regression objective (cross-entropy / log loss)
- KNN scaling + hyperparameters (k, distance)
- Baseline model choice trade-offs
- Decision trees: how leaf nodes make predictions

> **Goal:** write, run, and explain. Don’t just get the right number—be able to defend it in an interview.


---

## ✅ Setup (imports + reproducibility)


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

from sklearn.metrics import (
    mean_squared_error, r2_score,
    confusion_matrix, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, log_loss
)

np.random.seed(42)
plt.rcParams.update({"figure.dpi": 120})
print('✅ Imports loaded')


✅ Imports loaded


---

## 1) Data Leakage + Train/Val/Test

### Talking point
**Leakage** happens when your model sees information during training that would not be available at prediction time.
It inflates training/validation metrics and collapses on truly unseen data.

### Common leakage patterns
- Using `cancellation_date` to predict churn **before** cancellation happens
- Scaling/encoding using the full dataset (train+test) instead of fitting only on train
- Feature engineering using target-derived information (e.g., post-outcome aggregates)

### Correct pattern (interview-safe)
1) Split data
2) Fit preprocessing **only on train**
3) Apply to val/test
4) Tune hyperparameters on validation (or CV)
5) Report final once on test


In [2]:
# Mini example: leakage feature
df = pd.DataFrame({
    'tenure_months': [1, 2, 3, 4, 5, 6],
    'monthly_charges': [30, 35, 28, 80, 82, 85],
    # Leakage: this is only known AFTER churn happens
    'cancellation_date': ['2025-01-05', None, None, '2025-02-10', '2025-02-12', '2025-02-13'],
    'churn': [1, 0, 0, 1, 1, 1]
})
df


,tenure_months,monthly_charges,cancellation_date,churn
0,1,30,2025-01-05,1
1,2,35,NaN,0
2,3,28,NaN,0
3,4,80,2025-02-10,1
4,5,82,2025-02-12,1
5,6,85,2025-02-13,1


### Exercise 1 (TODO)
Decide which features are safe for a **one-month-ahead churn prediction**.

- Keep: features that exist **before** the outcome.
- Drop: features only known **after** the customer churns.

Fill in `safe_features`.


In [3]:
# # TODO: edit this list
# safe_features = ['tenure_months', 'monthly_charges']  # <-- TODO

# X = df[safe_features]
# y = df['churn']

# X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
# X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# print('Shapes:', X_train.shape, X_val.shape, X_test.shape)
# assert 'cancellation_date' not in safe_features, 'Leakage feature included!'
# print('✅ Leakage check passed')
from sklearn.model_selection import train_test_split

def safe_stratify_split(X, y, test_size, random_state=42):
    counts = y.value_counts()
    strat = y if (len(counts) >= 2 and counts.min() >= 2) else None
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=strat)

safe_features = ['tenure_months', 'monthly_charges']
X = df[safe_features].copy()
y = df['churn'].copy()

X_trainval, X_test, y_trainval, y_test = safe_stratify_split(X, y, test_size=0.2)
X_train, X_val, y_train, y_val = safe_stratify_split(X_trainval, y_trainval, test_size=0.25)

print('Shapes:', X_train.shape, X_val.shape, X_test.shape)
print("Class counts:",
      "\n train:", y_train.value_counts().to_dict(),
      "\n val:  ", y_val.value_counts().to_dict(),
      "\n test: ", y_test.value_counts().to_dict())

assert 'cancellation_date' not in safe_features, 'Leakage feature included!'
print('✅ Leakage check passed')

Shapes: (3, 2) (1, 2) (2, 2)
Class counts: 
 train: {1: 2, 0: 1} 
 val:   {1: 1} 
 test:  {1: 1, 0: 1}
✅ Leakage check passed


---

## 2) Scaling Leakage (the subtle one)

### Talking point
If you compute mean/std using **all data** (including test), the test distribution influences training. That’s leakage.

### Correct pattern
Use a `Pipeline` so scaling is fit only on training folds.


In [4]:
# Synthetic imbalanced dataset for classification
n = 2000
X_num = pd.DataFrame({
    'age': np.random.randint(18, 90, size=n),
    'income': np.random.lognormal(mean=11, sigma=0.4, size=n)  # big scale
})
# 1% positives, correlated slightly with high income
y = (X_num['income'] > np.quantile(X_num['income'], 0.99)).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X_num, y, test_size=0.2, random_state=42, stratify=y)

# ✅ Correct: scaler fit only on X_train inside the Pipeline
knn_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier(n_neighbors=15))
])

knn_pipe.fit(X_train, y_train)
y_prob = knn_pipe.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print('Confusion matrix:\n', confusion_matrix(y_test, y_pred))
print('Precision:', precision_score(y_test, y_pred, zero_division=0))
print('Recall   :', recall_score(y_test, y_pred, zero_division=0))


Confusion matrix:
 [[396   0]
 [  2   2]]
Precision: 1.0
Recall   : 0.5


---

## 3) Metrics for Imbalanced Data

### Talking point
With rare positives, **accuracy can be meaningless**. If 1% are positive, predicting all zeros gives 99% accuracy.

Prefer:
- **Recall** (if missing positives is costly)
- **Precision** (if false alarms are costly)
- **F1** (balance)
- **PR-AUC** (average precision) for rare-positive ranking quality

### Exercise 2 (TODO)
Compute accuracy, precision, recall, F1, and PR-AUC for the model above.


In [5]:
# TODO: compute and print the metrics
acc = (y_pred == y_test.values).mean()
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
prauc = average_precision_score(y_test, y_prob)

print({'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1, 'pr_auc': prauc})

assert 0 <= prauc <= 1
print('✅ Metrics computed')


{'accuracy': np.float64(0.995), 'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'pr_auc': 1.0}
✅ Metrics computed


---

## 4) Linear Regression: MSE and $R^2$ + Overfitting vs “bad labels”

### Talking point
- **MSE** measures average squared error (lower is better).
- **$R^2$** measures fraction of variance explained (closer to 1 is better on the same dataset).

Perfect training performance (MSE=0, $R^2$=1) but poor test performance usually means:
- **Overfitting** (model memorized training patterns)
- **Leakage** (target info slipped into features)
Not typically “wrong labels” unless you have evidence.

### Exercise 3 (TODO)
Fit linear regression on a noisy dataset and compare train vs test metrics.


In [6]:
# Create a noisy regression dataset
n = 300
X = pd.DataFrame({'x': np.random.uniform(-3, 3, size=n)})
y = 2.0 * X['x'] + 0.5 + np.random.normal(0, 1.0, size=n)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)

pred_train = lr.predict(X_train)
pred_test = lr.predict(X_test)

mse_train = mean_squared_error(y_train, pred_train)
r2_train = r2_score(y_train, pred_train)
mse_test = mean_squared_error(y_test, pred_test)
r2_test = r2_score(y_test, pred_test)

print({'mse_train': mse_train, 'r2_train': r2_train, 'mse_test': mse_test, 'r2_test': r2_test})
print('Model:', {'coef': float(lr.coef_[0]), 'intercept': float(lr.intercept_)})


{'mse_train': 0.8692520648860577, 'r2_train': 0.9314995852333344, 'mse_test': 0.7748686919986085, 'r2_test': 0.9370349267853199}
Model: {'coef': 1.9691225404066204, 'intercept': 0.4618605993044007}


---

## 5) Logistic Regression: Cross-Entropy (Log Loss)

### Talking point
Logistic regression predicts **probabilities** using the sigmoid:
$$p(y=1|x)=\sigma(w^Tx+b)$$
It is typically trained by minimizing **cross-entropy / log loss**:
- confident wrong predictions are punished heavily

### Exercise 4 (TODO)
Train logistic regression and compute log loss and PR-AUC.


In [7]:
from sklearn.datasets import make_classification

X, y = make_classification(
    n_samples=3000, n_features=6, n_informative=3, n_redundant=1,
    weights=[0.95, 0.05], flip_y=0.01, class_sep=1.0, random_state=42
)
X = pd.DataFrame(X, columns=[f'x{i}' for i in range(X.shape[1])])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

logreg = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000))
])
logreg.fit(X_train, y_train)

prob = logreg.predict_proba(X_test)[:, 1]
pred = (prob >= 0.5).astype(int)

# TODO: compute log loss + PR-AUC
ll = log_loss(y_test, prob)
prauc = average_precision_score(y_test, prob)

print('log_loss:', ll)
print('pr_auc  :', prauc)
print('precision:', precision_score(y_test, pred, zero_division=0))
print('recall   :', recall_score(y_test, pred, zero_division=0))


log_loss: 0.05774927286516142
pr_auc  : 0.886210589095148
precision: 1.0
recall   : 0.8


---

## 6) KNN Hyperparameters + Scaling

### Talking point
KNN depends on **distance**, so feature scales matter.

Key hyperparameters:
- `n_neighbors` (k): larger k → smoother boundary (lower variance, higher bias)
- `metric`: Euclidean vs Manhattan, etc.
- `weights`: uniform vs distance-weighted

### Exercise 5 (TODO)
Search over a few k values and pick the best using validation performance (PR-AUC).


In [8]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

k_values = [3, 5, 11, 21, 51]
results = []

for k in k_values:
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('knn', KNeighborsClassifier(n_neighbors=k))
    ])
    pipe.fit(X_train, y_train)
    prob_val = pipe.predict_proba(X_val)[:, 1]
    prauc_val = average_precision_score(y_val, prob_val)
    results.append({'k': k, 'pr_auc_val': prauc_val})

res_df = pd.DataFrame(results).sort_values('pr_auc_val', ascending=False)
res_df


,k,pr_auc_val
4,51,0.784264
3,21,0.783151
2,11,0.705723
1,5,0.705095
0,3,0.674097


---

## 7) Decision Trees: Leaf Nodes and Predictions

### Talking point
- **Classification tree:** a leaf predicts the **majority class** (and class probabilities from class frequencies).
- **Regression tree:** a leaf predicts the **mean target value** of training samples in that leaf.

### Exercise 6 (TODO)
Train a small decision tree and inspect a leaf prediction.


In [9]:
from sklearn.datasets import load_diabetes

# Regression tree demo
diab = load_diabetes(as_frame=True)
Xr = diab.data[['bmi', 'bp']].copy()
yr = diab.target

X_train, X_test, y_train, y_test = train_test_split(Xr, yr, test_size=0.25, random_state=42)

tree = DecisionTreeRegressor(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

pred = tree.predict(X_test)
print('MSE:', mean_squared_error(y_test, pred))
print('R2 :', r2_score(y_test, pred))

# Pick one sample and see its prediction
sample = X_test.iloc[[0]]
print('Sample features:', sample.to_dict(orient='records')[0])
print('Tree prediction:', float(tree.predict(sample)[0]))


MSE: 4141.756895166686
R2 : 0.2509965212270877
Sample features: {'bmi': -0.006205954135807083, 'bp': -0.015998975220305175}
Tree prediction: 140.52


---

## 8) Interview Drill: Explain in 30 seconds (write it)

Fill these in as short bullets you can say out loud.

### Prompt A — What is data leakage?
- 

### Prompt B — Why accuracy is misleading for 1% positives?
- 

### Prompt C — What does log loss measure?
- 

### Prompt D — How does increasing k change KNN behavior?
- 

### Prompt E — What does a regression tree leaf output?
- 


---

## 9) Reflection (2–4 sentences)

1) Which mistake in the quiz surprised you most?
2) What *new habit* will you use to prevent leakage?
3) Which metric will you lead with next time you see imbalanced data—and why?
